Isobel Parkes

19/06/26

Homework 07: Scraping with BeautifulSoup (and friends)

In [2]:
# Le Monde
# titles, subhead, article URL, whether it's premium or not, byline, article type, image URL
# make the CSV file auto-updating

# internet → requests → raw HTML string → BeautifulSoup → document

import requests
from bs4 import BeautifulSoup
import pandas as pd

response = requests.get("https://www.lemonde.fr/en/")
doc = BeautifulSoup(response.text, 'html.parser') # BeautifulSoup object which you can then query

In [3]:
# Le Monde uses article as the base class + modifier classes for type of article e.g. article--main, article--headlines, article--featured
# scrape all article types using the base class

stories = doc.find_all(class_="article")

# exploration of all articles 

for story in stories:
    row = {}
    classes = story.get('class', [])
    modifier = [c for c in classes if c.startswith('article--')]
    row['article_type'] = modifier[0] if modifier else 'unknown'

In [4]:
stories

[<div class="article article--main" data-zone="titles title_1"> <div class="lmd-link-clickarea"> <span class="icon__premium icon--outside" id="zone1-premium-96385"><span class="sr-only">Subscribers only</span></span><h1 class="article__title"><a aria-describedby="zone1-premium-96385" class="lmd-link-clickarea__link" href="https://www.lemonde.fr/en/environment/article/2026/06/20/is-air-conditioning-essential-for-coping-with-heatwaves-france-might-be-changing-its-mind-on-the-issue_6754695_114.html"> <p class="article__title-label">Is air conditioning essential for coping with heatwaves? France might be changing its mind on the issue</p> </a> </h1> <div class="article__media-container"> <picture class="article__media"> <img alt="A salesperson demonstrated home air conditioning systems to a couple at a Castorama store in Bondues, France, on May 26, 2026." height="2" loading="eager" sizes="(min-width: 1024px) 421px, 100vw" src="https://img.lemde.fr/2026/06/19/0/0/6000/4000/400/266/75/0/b471

In [5]:
print(stories[0].prettify())

<div class="article article--main" data-zone="titles title_1">
 <div class="lmd-link-clickarea">
  <span class="icon__premium icon--outside" id="zone1-premium-96385">
   <span class="sr-only">
    Subscribers only
   </span>
  </span>
  <h1 class="article__title">
   <a aria-describedby="zone1-premium-96385" class="lmd-link-clickarea__link" href="https://www.lemonde.fr/en/environment/article/2026/06/20/is-air-conditioning-essential-for-coping-with-heatwaves-france-might-be-changing-its-mind-on-the-issue_6754695_114.html">
    <p class="article__title-label">
     Is air conditioning essential for coping with heatwaves? France might be changing its mind on the issue
    </p>
   </a>
  </h1>
  <div class="article__media-container">
   <picture class="article__media">
    <img alt="A salesperson demonstrated home air conditioning systems to a couple at a Castorama store in Bondues, France, on May 26, 2026." height="2" loading="eager" sizes="(min-width: 1024px) 421px, 100vw" src="https://i

In [ ]:
# article__title [article__title-label, article__title--inline]
# article__desc for subhead
# article__byline
# <a> tag href for url, in featured, <a> tag is inside <h1> tag, with class="lmd-link-clickarea__link"
# icon__premium 
# article__media-container for image wrapper 
# article--TYPE (featured, main etc)

# this strategy is missing article cards, makes a df with 25 articles?

rows = []

for story in stories:
    row = {}

    try:
        row['title'] = story.select_one('h1, h2, h3').text.strip()
    except:
        try:
            # extract title from URL slug as fallback
            # e.g. ".../2026-world-cu...man-who-became-an-essential-football-intermediary-in-the-us_6754689_9.html"
            url = story.select_one('a')['href']
            slug = url.split('/')[-1]          # get last part of URL
            slug = slug.split('_')[0]          # remove trailing ID numbers
            row['title'] = slug.replace('-', ' ')  # hyphens to spaces
        except:
            print(f"Title not found")
            continue

    try:
        row['subhead'] = story.select_one('.article__desc').text.strip()
    except:
        print(f"Subhead not found: {row['title']}")

    try:
        row['url'] = story.select_one('a')['href']
    except:
        print(f"URL not found : {row['title']}")

    row['premium'] = story.find(class_='icon__premium') is not None

    try:
        row['byline'] = story.select_one('.article__byline').text.strip()
    except:
        print(f"Byline not found: {row['title']}")  # most cards don't have one

    try:
        row['article_type'] = story.select_one('.article__type').text.strip() # editorial label e.g. "Feature", "In Depth"
    except:
        classes = story.get('class', []) # CSS modifier class e.g. "article--main", "article--runner"
        modifier = [c for c in classes if c.startswith('article--')]
        row['article_type'] = modifier[0] if modifier else 'unknown'

    try:
        img = story.select_one('.article__media img')
        srcset = img.get('srcset', '')
        row['image_url'] = srcset.split(',')[0].strip().split(' ')[0] if srcset else img.get('src')
    except:
        print(f"Image not found: {row['title']}")

    rows.append(row)

Byline not found: Is air conditioning essential for coping with heatwaves? France might be changing its mind on the issue
Subhead not found: lebanon ceasefire between israel and hezbollah remains fragile under us iran pressure
Byline not found: lebanon ceasefire between israel and hezbollah remains fragile under us iran pressure
Subhead not found: german government stokes up the memory wars surrounding the displaced persons of 1945
Byline not found: german government stokes up the memory wars surrounding the displaced persons of 1945
Byline not found: thomas piketty to clarify its own stance and unite its ranks the left must propose a referendum to adopt a national solidarity tax
Byline not found: with spacex wall street demonstrates its unmatched capacity to fund innovation
Byline not found: fete de la musique our picks for concerts not to miss
Subhead not found: top ukrainian officials return polish awards in wwii row after zelensky stripped of highest honor
Byline not found: top ukr

In [7]:
df = pd.DataFrame(rows)
df.head()

,title,subhead,url,premium,article_type,image_url,byline
0,Is air conditioning essential for coping with ...,While more and more people are installing air ...,https://www.lemonde.fr/en/environment/article/...,True,article--main,https://img.lemde.fr/2026/06/19/0/0/6000/4000/...,NaN
1,lebanon ceasefire between israel and hezbollah...,NaN,https://www.lemonde.fr/en/international/articl...,True,article--headlines,https://img.lemde.fr/2026/06/19/0/0/8256/5504/...,NaN
2,german government stokes up the memory wars su...,NaN,https://www.lemonde.fr/en/international/articl...,True,article--headlines,https://img.lemde.fr/2026/06/19/0/0/5568/3712/...,NaN
3,thomas piketty to clarify its own stance and u...,As the 2027 presidential campaign draws nearer...,https://www.lemonde.fr/en/opinion/article/2026...,True,article--runner,https://img.lemde.fr/2026/01/24/0/0/4929/3286/...,NaN
4,with spacex wall street demonstrates its unmat...,"According to Goldman Sachs, American companies...",https://www.lemonde.fr/en/economy/article/2026...,True,article--runner,https://img.lemde.fr/2026/06/11/0/0/5515/3676/...,NaN


In [ ]:
df.to_csv('lemonde-articles.csv', index=False)